# Task

Prepare an end-to-end ETL with PySpark to

* extract data from the [kaggle spotify dataset](https://www.kaggle.com/datasets/kapturovalexander/spotify-data-from-pyspark-course/data)
* transform it (cleaning, calculate KPIs)
* and load results into local files

In [111]:
!pip install kagglehub

In [1]:
import kagglehub

# Download latest version
path_spotify_data = kagglehub.dataset_download(
    "kapturovalexander/spotify-data-from-pyspark-course"
)

print("Path to dataset files:", path_spotify_data)

Path to dataset files: /home/jovyan/.cache/kagglehub/datasets/kapturovalexander/spotify-data-from-pyspark-course/versions/53


## Extract

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import Window
import pyspark.sql.functions as sf
f = sf

In [3]:
spark = SparkSession.builder.appName("Spark-ETL-pipeline").getOrCreate()

In [4]:
#help(spark.read.csv)

In [5]:
df_err = (
    spark.read
    .csv(
        path_spotify_data,
        header=True,
        inferSchema=True,
    )
)

In [6]:
#df_spotify = df = spark.read.csv("data/spotify-data.csv", header=True, inferSchema=True)
df_spotify = df = (
    spark.read

    # alternative options only for testing
    .option("delimiter", ",")  # field delimiter, default: ','
    .option("sep", ",")        # field delimiter, default: ','
    
    .csv(
        path_spotify_data,
        header=True,
        inferSchema=True,
        sep=r',',          # only 'sep' works as parameter to csv(). 'delimiter' is not allowed (default: ',')
        quote=r'"',        # character used to quote any string which contains quote characters (default: '"')
        escape=r'"',       # unescape quotes within a quoted string escaped by doubling them as "" (default: '\')
        encoding="UTF-8",  # default: utf-8
    )
)

In [7]:
print(f"Number of rows: {df.count()}")

print("\nInferred Schema:")
df.printSchema()

print("\nPreview:")
df.show(5)

Number of rows: 169909

Inferred Schema:
root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- release_date: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- key: integer (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- explicit: integer (nullable = true)


Preview:
+--------------------+--------------------+--------------------+-----------+------------+----+------------+------------+------+----------------+--------+--------+-----------+-----

In [8]:
df.columns

['id',
 'name',
 'artists',
 'duration_ms',
 'release_date',
 'year',
 'acousticness',
 'danceability',
 'energy',
 'instrumentalness',
 'liveness',
 'loudness',
 'speechiness',
 'tempo',
 'valence',
 'mode',
 'key',
 'popularity',
 'explicit']

### Explore extracted raw data from CSV file

Initially the `escape='"'` was missing leading to misinterpretation of
some (not all) string values containing doubly quotes (")
escaped by doubling them (""). The following snippets were used
to skim through the data and find columns not imported correctly.

In [9]:
# with the wrong escape the later columns will have parts of the
# name/artist columns
res = df_err.select(f.col("year")).distinct()
res.show(5)
res.take(5)

+-----------------+
|             year|
+-----------------+
|           164466|
|           1/1/53|
| 'Georges Prêtre'|
|           1/1/08|
|            28600|
+-----------------+
only showing top 5 rows



[Row(year='164466'),
 Row(year='1/1/53'),
 Row(year=" 'Georges Prêtre'"),
 Row(year='1/1/08'),
 Row(year='28600')]

In [10]:
# expected result
res = df_spotify.select(f.col("year")).distinct()
res.show(5)
res.take(5)

+----+
|year|
+----+
|1959|
|1990|
|1975|
|1977|
|2003|
+----+
only showing top 5 rows



[Row(year=1959),
 Row(year=1990),
 Row(year=1975),
 Row(year=1977),
 Row(year=2003)]

In [11]:
[r.year for r in df_err.filter(f.col("year").contains("Georges")).collect()]

[" 'Georges Prêtre'",
 "['Georges Bizet', 'Victoria de los Ángeles', 'Orchestre National Radiodiffusion Française', 'Sir Thomas Beecham', 'Orchestre National de la Radiodiffusion-Television Francaise']",
 " 'Georges Prêtre'"]

In [12]:
# manually skim through data rows to see what is wrong
df = df_err
row = 395 -2
range = 5  # 100
# +2 to i because +1 for header +1 for python counting from 0
[(i+row +2,
  r.id[:5],
  r.name[:5],
  r.name[-5:],
  f"{len(r.name):05d}",
  r.artists[:5],
  r.artists[-5:],
  r.year,
 ) for i,r in enumerate(df.collect()[row : row+range])]

[(395, '2Rtcf', 'Viola', 'agoas', '00017', "['Jar", "nho']", '1940'),
 (396, '2S0wn', 'O xem', 'agkas', '00010', "['Rit", "tzi']", '1940'),
 (397, '2SDRs', 'Oi om', 'deias', '00024', "['Roz", "ino']", '1940'),
 (398, '2SEGa', '"Libu', 'ibuše', '00049', ' Přem', 'tadic', ' Krasava'),
 (399, '2SFh3', 'Ta di', 'matia', '00016', "['Ste", "dis']", '1940')]

In [13]:
# add row number without changing order of original data
# f.lit(1) is the literal 1 and therefore constant for sorting
w = Window.orderBy(f.lit(1))  # 
df_err = (
    df_err
    .withColumn("rn", f.row_number().over(w))
)

In [14]:
row = df_err.filter(f.col("rn") == 397).first()
print(f"Full row:\n{row}")
print()
print("row number in csv:", row.rn, "(ignoring header row)")
print("row.id:           ", row.id)
print("repr(row.name):  ", repr(row.name))

Full row:
Row(id='2SEGaosrnf7rpEtPAldV2e', name='"Libuše, Act III, Scene 3: ""Buď vítaná"" (Libuše', artists=' Přemysl ze Stadic', duration_ms=' Chrudoš od Otavy', release_date=' Šťáhlav na Radbuze', year=' Krasava', acousticness=' Radmila', danceability=' Lutobor', energy=' Radovan) - Live"', instrumentalness="['Bedřich Smetana', 'Marie Podvalová', 'Stanislav Muž', 'Vilém Zítek', 'Josef Vojta', 'Ota Horáková', 'Marta Krásová', 'Jaroslav Veverka', 'Prague National Theatre Chorus', 'Prague National Theatre Orchestra', 'Jan Maria Ouředník', 'Václav Talich', 'Josef Křikava']", liveness='60507', loudness='1940', speechiness='1940', tempo='0.991', valence='0.449', mode='0.315', key='0.137', popularity='0.145', explicit='-15.779', rn=397)

row number in csv: 397 (ignoring header row)
row.id:            2SEGaosrnf7rpEtPAldV2e
repr(row.name):   '"Libuše, Act III, Scene 3: ""Buď vítaná"" (Libuše'


In [15]:
row = df_err.filter(f.col("rn") == 351).first()
print(f"Full row:\n{row}")
print()
print("row number in csv:", row.rn, "(ignoring header row)")
print("row.id:           ", row.id)
print("repr(row.name):  ", repr(row.name))

Full row:
Row(id='7n464SNnIopih8qPrtmTIE', name='"Symphony No. 6 in F Major, Op. 68 ""Pastoral"": III. Lustiges Zusammensein der Landleute (Allegro)"', artists="['Ludwig van Beethoven', 'Concertgebouworkest', 'Erich Kleiber']", duration_ms='313800', release_date='1936', year='1936', acousticness='0.956', danceability='0.363', energy='0.122', instrumentalness='0.327', liveness='0.206', loudness='-16.58', speechiness='0.0591', tempo='154.33', valence='0.428', mode='1', key='5', popularity='0', explicit='0', rn=351)

row number in csv: 351 (ignoring header row)
row.id:            7n464SNnIopih8qPrtmTIE
repr(row.name):   '"Symphony No. 6 in F Major, Op. 68 ""Pastoral"": III. Lustiges Zusammensein der Landleute (Allegro)"'


### We try to cast the 'year' column to int to find values where this fails as NULL

In [16]:
from pyspark.sql.types import IntegerType
_df = (
    df_err
    .withColumn(
        "year_int",
        f.col("year").cast(IntegerType())
    ).select(
        f.col("id"),
        f.col("year"),
        f.col("year_int"),
    )
    .distinct()
    .filter(f.col("year_int").isNull())
    .filter(f.col("id").startswith("2"))
    #.orderBy(f.col("year").desc())
    #.orderBy(f.col("id"))
)
_df.show()

+--------------------+--------------------+--------+
|                  id|                year|year_int|
+--------------------+--------------------+--------+
|2FQGMcSQxgJXKccBe...|              1/1/83|    NULL|
|2t5qPMScyyl7f1SaE...|              1/1/56|    NULL|
|2kT9dKUxXRLLQLESb...|             Krasava|    NULL|
|2iYM4CGkoTfsLw9MI...|              1/1/00|    NULL|
|2gU9IcWtk5LzEsrQt...|   'Lorraine Geller'|    NULL|
|22Cp6o8MHBJrJKmgZ...|          Marchesa]"|    NULL|
|2f9cXYQSkeHZD5oAB...|              1/1/57|    NULL|
|2yTjLx8CDh1kpfl9L...|              1/1/72|    NULL|
|27ICzXFf4UqDFidh4...|           Amneris]"|    NULL|
|28Q76S6AAv1Vg60xJ...|            Nannetta|    NULL|
|2b9M9JzeYCVts2PMJ...|              1/1/49|    NULL|
|26wcBprDh6EiILdg1...|['Giuseppe Verdi'...|    NULL|
|25WtkgYzdYCR0n1Is...|              1/1/57|    NULL|
|2OaPyH3JLuft6iuNF...|             'L-Boy'|    NULL|
|21RxocdHYXogqYWST...|         Cavaliere]"|    NULL|
|2rrjt9urJRGiV83jw...|  Šťáhlav na Radbuze|   

### Specifying the schema

[Data types reference](https://spark.apache.org/docs/latest/sql-ref-datatypes.html)

In [27]:
df = df_spotify

In [28]:
# different release_date formats
res = (
    df_spotify
    .select(f.col("release_date"))
    .distinct()
    .orderBy(f.col("release_date"))
)
print(
", ".join(
[r.release_date 
 for r in res.collect()
 if "/" in r.release_date
][:5])
)
print(
", ".join(
[r.release_date 
 for r in res.collect()
 if "-" in r.release_date
][:5])
)
print(
", ".join(
[r.release_date 
 for r in res.collect()
 if (not "-" in r.release_date and not "/" in r.release_date)
][:5])
)

1/1/00, 1/1/01, 1/1/02, 1/1/03, 1/1/04
1956-03, 1956-04, 1957-03, 1957-04, 1957-06
1921, 1922, 1923, 1924, 1925


In [31]:
", ".join(
    str(r.year) for r in
    df.select(f.col("year"))
    .distinct().orderBy(f.col("year"))
    .collect()
)

'1921, 1922, 1923, 1924, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938, 1939, 1940, 1941, 1942, 1943, 1944, 1945, 1946, 1947, 1948, 1949, 1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020'

In [32]:
res = (
    df_spotify
    .select(f.col("explicit"))
    .distinct()
)
res.show()

+--------+
|explicit|
+--------+
|       1|
|       0|
+--------+



In [18]:
df.columns

['id',
 'name',
 'artists',
 'duration_ms',
 'release_date',
 'year',
 'acousticness',
 'danceability',
 'energy',
 'instrumentalness',
 'liveness',
 'loudness',
 'speechiness',
 'tempo',
 'valence',
 'mode',
 'key',
 'popularity',
 'explicit']

In [19]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    FloatType,
    DateType,
)

schema = StructType([
    StructField("id", StringType(), nullable=False),
    
    StructField("name", StringType(), nullable=True),
    StructField("artists", StringType(), nullable=True),
    StructField("duration_ms", IntegerType(), nullable=True),
    
    StructField("release_date", StringType(), nullable=True),
    #StructField("release_date", DateType(), nullable=True),
    
    StructField("year", IntegerType(), nullable=True),
    StructField("acousticness", FloatType(), nullable=True),
    StructField("danceability", FloatType(), nullable=True),
    StructField("energy", FloatType(), nullable=True),
    StructField("instrumentalness", FloatType(), nullable=True),
    StructField("liveness", FloatType(), nullable=True),
    StructField("loudness", FloatType(), nullable=True),
    StructField("speechiness", FloatType(), nullable=True),
    StructField("tempo", FloatType(), nullable=True),
    StructField("valence", FloatType(), nullable=True),
    StructField("mode", IntegerType(), nullable=True),
    StructField("key", IntegerType(), nullable=True),
    StructField("popularity", FloatType(), nullable=True),
    StructField("explicit", IntegerType(), nullable=True),
])